# Lab 05.2 — Spark e MongoDB (database documentale)

MongoDB memorizza **documenti JSON** (BSON) con struttura flessibile e annidata.
Il **MongoDB Spark Connector** permette di:

* leggere una collezione come DataFrame (lo schema viene **dedotto** da un campione di documenti);
* spingere filtri e *aggregation pipeline* verso MongoDB;
* scrivere DataFrame (anche con campi annidati e array) come documenti.

In [ ]:
import os
from pyspark.sql import SparkSession, functions as F

MONGO = os.environ.get("MONGO_URI", "mongodb://mongo:27017")

spark = (SparkSession.builder
         .appName("Lab05-MongoDB")
         .master(os.environ.get("SPARK_MASTER", "local[*]"))
         .config("spark.jars.packages", "org.mongodb.spark:mongo-spark-connector_2.13:11.0.2")
         .config("spark.jars.ivy", "/home/jovyan/.ivy2")
         .config("spark.mongodb.read.connection.uri", MONGO)
         .config("spark.mongodb.write.connection.uri", MONGO)
         .getOrCreate())

def mongo(collezione, db="shop"):
    """Lettore per una collezione."""
    return spark.read.format("mongodb").option("database", db).option("collection", collezione)

## 1. Scrivere documenti annidati
Le colonne di tipo `struct` diventano sotto-documenti, gli `array` diventano array BSON.

In [ ]:
recensioni = spark.createDataFrame([
    (1, "Cliente_001", {"titolo": "Il Signore degli Anelli", "categoria": "Libri"}, 5, ["epico", "classico"]),
    (2, "Cliente_002", {"titolo": "Cuffie Bluetooth",        "categoria": "Elettronica"}, 3, ["audio"]),
    (3, "Cliente_001", {"titolo": "Cuffie Bluetooth",        "categoria": "Elettronica"}, 4, ["audio", "economico"]),
    (4, "Cliente_003", {"titolo": "Pallone da calcio",       "categoria": "Sport"}, 5, []),
    (5, "Cliente_004", {"titolo": "Il Signore degli Anelli", "categoria": "Libri"}, 4, ["lungo"]),
], "id INT, cliente STRING, prodotto STRUCT<titolo: STRING, categoria: STRING>, voto INT, tag ARRAY<STRING>")

(recensioni.write.format("mongodb").mode("overwrite")
           .option("database", "shop").option("collection", "recensioni").save())
print("Scritti", recensioni.count(), "documenti")

## 2. Leggere: schema dedotto dai documenti

In [ ]:
rec = mongo("recensioni").load()
rec.printSchema()
rec.show(truncate=False)

## 3. Interrogare campi annidati e array

In [ ]:
(rec.where(F.col("prodotto.categoria") == "Elettronica")
    .select("cliente", "prodotto.titolo", "voto").show())

(rec.select(F.explode("tag").alias("tag"))
    .groupBy("tag").count().orderBy(F.desc("count")).show())

rec.createOrReplaceTempView("recensioni")
spark.sql("""
    SELECT prodotto.titolo AS titolo, ROUND(AVG(voto), 2) AS voto_medio, COUNT(*) AS n
    FROM recensioni GROUP BY prodotto.titolo ORDER BY voto_medio DESC
""").show(truncate=False)

## 4. Pushdown verso MongoDB
* I **filtri** del DataFrame diventano uno stadio `$match` eseguito da MongoDB.
* Con l'opzione `aggregation.pipeline` si può far eseguire a MongoDB un'intera pipeline di aggregazione.

In [ ]:
# Il filtro viene inviato a MongoDB come {"$match": {"voto": {"$gte": 4}}}
rec.where("voto >= 4").select("cliente", "voto").show()

pipeline = """[
  {"$unwind": "$tag"},
  {"$group": {"_id": "$tag", "n": {"$sum": 1}, "voto_medio": {"$avg": "$voto"}}},
  {"$sort": {"n": -1}}
]"""
mongo("recensioni").option("aggregation.pipeline", pipeline).load().show()

## 5. Un caso tipico: da DataFrame "piatto" a documenti
Raggruppiamo righe per cliente e salviamo un documento per cliente con l'elenco dei suoi acquisti (struttura naturale per MongoDB).

In [ ]:
acquisti = spark.createDataFrame([
    ("Cliente_001", "2025-01-10", "Libri", 25.0), ("Cliente_001", "2025-02-03", "Sport", 40.0),
    ("Cliente_002", "2025-01-15", "Elettronica", 120.0), ("Cliente_001", "2025-03-01", "Libri", 15.5),
    ("Cliente_003", "2025-02-20", "Casa", 60.0), ("Cliente_002", "2025-03-11", "Libri", 18.0),
], "cliente STRING, data STRING, categoria STRING, importo DOUBLE")

profili = (acquisti.groupBy("cliente")
           .agg(F.count("*").alias("n_acquisti"),
                F.round(F.sum("importo"), 2).alias("totale"),
                F.collect_list(F.struct("data", "categoria", "importo")).alias("acquisti"))
           .withColumnRenamed("cliente", "_id"))       # _id = chiave del documento

profili.write.format("mongodb").mode("overwrite").option("database", "shop").option("collection", "profili").save()
mongo("profili").load().show(truncate=False)

In [ ]:
spark.stop()